# 1\.2\.1 Standardize Temporal Representations

This notebook builds the shared temporal contract that each mobility dataset needs before spatial alignment and multimodal integration\. The main job is to preserve each source's real time structure while adding the same core fields everywhere: date, weekday, hour, temporal bucket, congestion\-pricing period, and deterministic ordering fields for downstream sorting and feature engineering\. The outputs stay modality\-specific here and feed directly into \`1\.2\.2\` for spatial standardization\.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

import duckdb  # For those huge parquet files!
import pyarrow.parquet as pq
import gc

Note: Across modalities, this notebook is standardizing toward the same core temporal fields: date where available or reconstructable, year, month, day\_of\_week, hour, daypart, daypart\_order, temporal\_bucket, temporal\_bucket\_order, observation\_sequence\_id where appropriate, and pre\_post\_cp\. Some source datasets do not natively support every field\. In those cases, this notebook should either reconstruct the field with explicit provenance, as with Bus profile dates, or document why the field is not available\. Feature engineering should still happen later, but deterministic temporal ordering fields are part of the foundation contract\.

In [2]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")

TRAFFIC_COUNTS_STAGED_PATH = PIPELINE_DATA_DIR / "1.1.1.traffic_counts_staged.parquet"
TRAFFIC_COUNTS_TEMPORAL_PATH = PIPELINE_DATA_DIR / "1.2.1.traffic_counts_temporal.parquet"

## 1\.2\.1\.1 Add temporal fields to traffic data

In [3]:
# ---------------------------------------------------------------------
# Load staged traffic counts
# ---------------------------------------------------------------------

traffic_counts = pd.read_parquet(TRAFFIC_COUNTS_STAGED_PATH)

traffic_counts.head()

,segment_id,borough,street_name,from_street,to_street,direction,timestamp,traffic_volume,geometry_wkt,observation_count
0,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:00:00,16,POINT (932655.1292010084 168977.97581364735),1
1,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:15:00,19,POINT (932655.1292010084 168977.97581364735),1
2,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:30:00,10,POINT (932655.1292010084 168977.97581364735),1
3,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:45:00,12,POINT (932655.1292010084 168977.97581364735),1
4,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 01:00:00,16,POINT (932655.1292010084 168977.97581364735),1


In [4]:
# ---------------------------------------------------------------------
# Shared temporal standardization helpers
# ---------------------------------------------------------------------

STUDY_START_DATE = pd.Timestamp("2023-01-01")
STUDY_END_DATE = pd.Timestamp("2026-03-31")
CONGESTION_PRICING_START_DATE = pd.Timestamp("2025-01-05")

DAYPART_ORDER = {
    "overnight": 1,
    "am_peak": 2,
    "midday": 3,
    "pm_peak": 4,
    "evening": 5,
}

TEMPORAL_BUCKET_ORDER = {
    "weekday_overnight": 1,
    "weekday_am_peak": 2,
    "weekday_midday": 3,
    "weekday_pm_peak": 4,
    "weekday_evening": 5,
    "weekend_overnight": 1,
    "weekend_am_peak": 2,
    "weekend_midday": 3,
    "weekend_pm_peak": 4,
    "weekend_evening": 5,
}

VALID_TEMPORAL_BUCKETS = set(TEMPORAL_BUCKET_ORDER)
VALID_PRE_POST_CP_VALUES = {"pre_cp", "post_cp"}


def assign_daypart(hour):
    """Assign the project-standard daypart from an hour-of-day value."""
    if pd.isna(hour):
        return np.nan

    hour = int(hour)

    if 0 <= hour < 6:
        return "overnight"
    if 6 <= hour < 10:
        return "am_peak"
    if 10 <= hour < 16:
        return "midday"
    if 16 <= hour < 20:
        return "pm_peak"
    if 20 <= hour < 24:
        return "evening"
    return np.nan


def assign_temporal_bucket(day_of_week, hour):
    """Assign the canonical temporal bucket label without changing legacy labels."""
    daypart = assign_daypart(hour)

    if pd.isna(daypart) or pd.isna(day_of_week):
        return np.nan

    day_type = "weekend" if day_of_week in ["Saturday", "Sunday"] else "weekday"
    return f"{day_type}_{daypart}"


def add_temporal_ordering_fields(df, date_col="date"):
    """Add daypart, bucket order, and optional observation sequence fields."""
    df = df.copy()

    df["daypart"] = df["hour"].apply(assign_daypart)
    df["daypart_order"] = df["daypart"].map(DAYPART_ORDER).astype("Int64")
    df["temporal_bucket_order"] = (
        df["temporal_bucket"].map(TEMPORAL_BUCKET_ORDER).astype("Int64")
    )

    if date_col in df.columns:
        date_values = pd.to_datetime(df[date_col], errors="coerce")
        df["observation_sequence_id"] = (
            (date_values - STUDY_START_DATE).dt.days * len(DAYPART_ORDER)
            + df["daypart_order"]
        ).astype("Int64")

    return df


def assign_pre_post_cp(date_values):
    """Assign the project-standard congestion-pricing period labels."""
    normalized_dates = pd.to_datetime(date_values, errors="coerce")
    return np.where(
        normalized_dates >= CONGESTION_PRICING_START_DATE,
        "post_cp",
        "pre_cp",
    )


def filter_to_study_period(df, date_col="date"):
    """Filter rows to the project study period using an explicit date column."""
    df = df.copy()
    date_values = pd.to_datetime(df[date_col], errors="coerce")
    in_study_period = date_values.between(STUDY_START_DATE, STUDY_END_DATE)
    return df.loc[in_study_period].copy()


def validate_temporal_contract(df, dataset_name, require_date=True):
    """Run lightweight temporal contract checks for a standardized output."""
    required_columns = [
        "year",
        "month",
        "day_of_week",
        "hour",
        "daypart",
        "daypart_order",
        "temporal_bucket",
        "temporal_bucket_order",
        "pre_post_cp",
    ]

    if require_date:
        required_columns.extend(["date", "observation_sequence_id"])

    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"{dataset_name} missing temporal columns: {missing_columns}")

    unexpected_buckets = sorted(set(df["temporal_bucket"].dropna()) - VALID_TEMPORAL_BUCKETS)
    if unexpected_buckets:
        raise ValueError(f"{dataset_name} has unexpected temporal buckets: {unexpected_buckets}")

    unexpected_cp_values = sorted(set(df["pre_post_cp"].dropna()) - VALID_PRE_POST_CP_VALUES)
    if unexpected_cp_values:
        raise ValueError(f"{dataset_name} has unexpected pre_post_cp values: {unexpected_cp_values}")

    invalid_hours = ~df["hour"].between(0, 23)
    if invalid_hours.any():
        raise ValueError(f"{dataset_name} has {invalid_hours.sum():,} invalid hour values")

    if require_date:
        date_values = pd.to_datetime(df["date"], errors="coerce")
        out_of_range = ~date_values.between(STUDY_START_DATE, STUDY_END_DATE)
        if out_of_range.any():
            raise ValueError(f"{dataset_name} has {out_of_range.sum():,} rows outside the study period")

    print(f"{dataset_name} temporal contract checks passed.")

In [5]:
# ---------------------------------------------------------------------
# Add standardized temporal fields to traffic counts
# ---------------------------------------------------------------------

traffic_temporal = traffic_counts.copy()

traffic_temporal["timestamp"] = pd.to_datetime(
    traffic_temporal["timestamp"],
    errors="coerce"
)
traffic_temporal["date"] = traffic_temporal["timestamp"].dt.normalize()
traffic_temporal = filter_to_study_period(traffic_temporal, date_col="date")

traffic_temporal["year"] = traffic_temporal["timestamp"].dt.year
traffic_temporal["month"] = traffic_temporal["timestamp"].dt.month
traffic_temporal["day_of_week"] = traffic_temporal["timestamp"].dt.day_name()
traffic_temporal["hour"] = traffic_temporal["timestamp"].dt.hour

traffic_temporal["temporal_bucket"] = traffic_temporal.apply(
    lambda row: assign_temporal_bucket(row["day_of_week"], row["hour"]),
    axis=1
)
traffic_temporal = add_temporal_ordering_fields(traffic_temporal, date_col="date")

traffic_temporal["pre_post_cp"] = assign_pre_post_cp(traffic_temporal["date"])
traffic_temporal["date"] = pd.to_datetime(traffic_temporal["date"]).dt.date
traffic_temporal["temporal_grain_type"] = "native_timestamp"
traffic_temporal["date_assignment_method"] = "derived_from_native_timestamp"

validate_temporal_contract(traffic_temporal, "Traffic Counts")

traffic_temporal.head()

Traffic Counts temporal contract checks passed.


,segment_id,borough,street_name,from_street,to_street,direction,timestamp,traffic_volume,geometry_wkt,observation_count,...,day_of_week,hour,temporal_bucket,daypart,daypart_order,temporal_bucket_order,observation_sequence_id,pre_post_cp,temporal_grain_type,date_assignment_method
0,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:00:00,16,POINT (932655.1292010084 168977.97581364735),1,...,Wednesday,0,weekday_overnight,overnight,1,1,1241,pre_cp,native_timestamp,derived_from_native_timestamp
1,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:15:00,19,POINT (932655.1292010084 168977.97581364735),1,...,Wednesday,0,weekday_overnight,overnight,1,1,1241,pre_cp,native_timestamp,derived_from_native_timestamp
2,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:30:00,10,POINT (932655.1292010084 168977.97581364735),1,...,Wednesday,0,weekday_overnight,overnight,1,1,1241,pre_cp,native_timestamp,derived_from_native_timestamp
3,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:45:00,12,POINT (932655.1292010084 168977.97581364735),1,...,Wednesday,0,weekday_overnight,overnight,1,1,1241,pre_cp,native_timestamp,derived_from_native_timestamp
4,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 01:00:00,16,POINT (932655.1292010084 168977.97581364735),1,...,Wednesday,1,weekday_overnight,overnight,1,1,1241,pre_cp,native_timestamp,derived_from_native_timestamp


In [6]:
# ---------------------------------------------------------------------
# Validate standardized temporal fields
# ---------------------------------------------------------------------

temporal_fields = [
    "timestamp",
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
    "temporal_grain_type",
    "date_assignment_method",
]

temporal_qa = pd.DataFrame({
    "field": temporal_fields,
    "missing_count": [traffic_temporal[col].isna().sum() for col in temporal_fields],
    "missing_rate": [traffic_temporal[col].isna().mean() for col in temporal_fields],
    "unique_values": [traffic_temporal[col].nunique(dropna=True) for col in temporal_fields],
})

display(temporal_qa)

print("Traffic date range:")
display(pd.DataFrame([{
    "min_date": traffic_temporal["date"].min(),
    "max_date": traffic_temporal["date"].max(),
    "distinct_dates": traffic_temporal["date"].nunique(),
}]))

print("Temporal bucket distribution:")
display(traffic_temporal["temporal_bucket"].value_counts(dropna=False).sort_index())

print("Pre/post congestion pricing distribution:")
display(traffic_temporal["pre_post_cp"].value_counts(dropna=False))

print("Temporal provenance distribution:")
display(
    traffic_temporal[["temporal_grain_type", "date_assignment_method"]]
    .value_counts(dropna=False)
    .reset_index(name="count")
)

,field,missing_count,missing_rate,unique_values
0,timestamp,0,0.0,68452
1,date,0,0.0,714
2,year,0,0.0,4
3,month,0,0.0,12
4,day_of_week,0,0.0,7
5,hour,0,0.0,24
6,daypart,0,0.0,5
7,daypart_order,0,0.0,5
8,temporal_bucket,0,0.0,10
9,temporal_bucket_order,0,0.0,5


Traffic date range:


,min_date,max_date,distinct_dates
0,2023-01-06,2026-02-03,714


Temporal bucket distribution:


temporal_bucket
weekday_am_peak      31214
weekday_evening      31201
weekday_midday       46823
weekday_overnight    46820
weekday_pm_peak      31215
weekend_am_peak      13231
weekend_evening      13245
weekend_midday       19848
weekend_overnight    19821
weekend_pm_peak      13233
Name: count, dtype: int64

Pre/post congestion pricing distribution:


pre_post_cp
pre_cp     175163
post_cp     91488
Name: count, dtype: int64

Temporal provenance distribution:


,temporal_grain_type,date_assignment_method,count
0,native_timestamp,derived_from_native_timestamp,266651


Findings\. Temporal standardization completed cleanly for the traffic dataset\. All standardized temporal fields were populated without missing values, with full coverage across dates, hours, weekdays, temporal buckets, and pre/post congestion\-pricing periods\.

In [7]:
# ---------------------------------------------------------------------
# Write temporal-standardized traffic counts
# ---------------------------------------------------------------------

traffic_temporal.to_parquet(
    TRAFFIC_COUNTS_TEMPORAL_PATH,
    index=False
)

print(f"Wrote temporal-standardized traffic counts to: {TRAFFIC_COUNTS_TEMPORAL_PATH}")
print(f"Rows written: {len(traffic_temporal):,}")

Wrote temporal-standardized traffic counts to: pipeline_data/1.2.1.traffic_counts_temporal.parquet
Rows written: 266,651


## 1\.2\.1\.2 Add temporal fields to bus data

This section standardizes the temporal representation of the bus dataset using the Year, Month, Day of Week, and Hour of Day fields identified during dataset inspection\. Unlike Traffic, Subway, Taxi, and Bridge/Tunnel data, Bus speeds are profile\-style observations at Year × Month × Day\-of\-Week × Hour × Route Segment grain rather than true daily observations\. To avoid carrying an ambiguous January 2025 congestion\-pricing assignment downstream, this section reconstructs calendar dates within each year/month that match the source day\-of\-week label\. The resulting Bus temporal output carries exact \`date\`, \`pre\_post\_cp\`, daypart, and temporal ordering fields while preserving provenance columns that identify the dates as calendar\-expanded profile dates rather than native daily observations\.

In [8]:
# ---------------------------------------------------------------------
# Configure bus temporal standardization paths
# ---------------------------------------------------------------------

BUS_STAGED_DIR = PIPELINE_DATA_DIR / "1.1.4 bus_route_segment_speeds_parquet"
BUS_TEMPORAL_DIR = PIPELINE_DATA_DIR / "1.2.1.bus_speeds_temporal_parquet"

BUS_TEMPORAL_DIR.mkdir(parents=True, exist_ok=True)

bus_staged_files = sorted(BUS_STAGED_DIR.glob("*.parquet"))

print(f"Found {len(bus_staged_files)} staged bus parquet files.")
print(f"Temporal-standardized bus output directory: {BUS_TEMPORAL_DIR}")

Found 39 staged bus parquet files.
Temporal-standardized bus output directory: pipeline_data/1.2.1.bus_speeds_temporal_parquet


In [9]:
# ---------------------------------------------------------------------
# Add standardized temporal fields to bus parquet partitions
# ---------------------------------------------------------------------

REBUILD_BUS_TEMPORAL = False

BUS_REQUIRED_TEMPORAL_COLUMNS = [
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
    "temporal_grain_type",
    "date_assignment_method",
]


def build_bus_calendar_lookup(study_start_date, study_end_date):
    """Build date rows used to expand monthly day-of-week bus profiles."""
    calendar = pd.DataFrame({
        "date": pd.date_range(study_start_date, study_end_date, freq="D")
    })
    calendar["year"] = calendar["date"].dt.year.astype("int64")
    calendar["month"] = calendar["date"].dt.month.astype("int64")
    calendar["day_of_week"] = calendar["date"].dt.day_name()
    return calendar


bus_calendar_lookup = build_bus_calendar_lookup(
    STUDY_START_DATE,
    STUDY_END_DATE,
)

bus_temporal_files = sorted(BUS_TEMPORAL_DIR.glob("*.parquet"))

existing_bus_temporal_is_current = False
if bus_temporal_files:
    sample_schema = pq.read_schema(bus_temporal_files[0])
    existing_columns = set(sample_schema.names)
    existing_bus_temporal_is_current = set(BUS_REQUIRED_TEMPORAL_COLUMNS).issubset(
        existing_columns
    )

if bus_temporal_files and existing_bus_temporal_is_current and not REBUILD_BUS_TEMPORAL:
    print("Temporal-standardized bus files already exist with the current schema.")
    print("Set REBUILD_BUS_TEMPORAL = True to regenerate them.")

else:
    if bus_temporal_files:
        print("Existing bus temporal files are missing the current schema or rebuild was requested.")
        print("Regenerating bus temporal parquet files with the current schema...")

    for input_path in bus_staged_files:
        output_path = BUS_TEMPORAL_DIR / input_path.name

        df = pd.read_parquet(input_path)

        # Preserve source profile fields while creating common temporal names.
        df["year"] = df["Year"].astype("int64")
        df["month"] = df["Month"].astype("int64")
        df["day_of_week"] = df["Day of Week"]
        df["hour"] = df["Hour of Day"].astype("int64")

        df["temporal_bucket"] = df.apply(
            lambda row: assign_temporal_bucket(row["day_of_week"], row["hour"]),
            axis=1
        )
        df = add_temporal_ordering_fields(df, date_col=None)

        # Bus speeds are monthly day-of-week profiles. Expand each profile row
        # to the calendar dates in that month that match its day-of-week label
        # so the CP boundary is exact and downstream panels have true dates.
        df = df.merge(
            bus_calendar_lookup,
            on=["year", "month", "day_of_week"],
            how="inner",
            validate="many_to_many",
        )

        df["pre_post_cp"] = assign_pre_post_cp(df["date"])
        df["observation_sequence_id"] = (
            (df["date"] - STUDY_START_DATE).dt.days * len(DAYPART_ORDER)
            + df["daypart_order"]
        ).astype("int64")
        df["date"] = pd.to_datetime(df["date"]).dt.date

        df["temporal_grain_type"] = "calendar_expanded_profile"
        df["date_assignment_method"] = "expanded_from_year_month_day_of_week"

        validate_temporal_contract(df, f"Bus Speeds {input_path.name}")

        df.to_parquet(output_path, index=False)

        print(f"Wrote: {output_path.name} ({len(df):,} rows)")

        del df
        gc.collect()

Temporal-standardized bus files already exist with the current schema.
Set REBUILD_BUS_TEMPORAL = True to regenerate them.


In [10]:
# ---------------------------------------------------------------------
# Validate standardized temporal fields for bus data
# ---------------------------------------------------------------------

bus_temporal_files = sorted(BUS_TEMPORAL_DIR.glob("*.parquet"))

temporal_fields = [
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
    "temporal_grain_type",
    "date_assignment_method",
]

qa_records = []
bucket_counts = []
pre_post_counts = []
date_ranges = []
provenance_counts = []

for path in bus_temporal_files:
    df = pd.read_parquet(path, columns=temporal_fields)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    for col in temporal_fields:
        qa_records.append({
            "file": path.name,
            "field": col,
            "rows": len(df),
            "missing_count": df[col].isna().sum(),
            "unique_values": df[col].nunique(dropna=True),
        })

    bucket_counts.append(
        df["temporal_bucket"]
        .value_counts(dropna=False)
        .rename_axis("temporal_bucket")
        .reset_index(name="count")
    )

    pre_post_counts.append(
        df["pre_post_cp"]
        .value_counts(dropna=False)
        .rename_axis("pre_post_cp")
        .reset_index(name="count")
    )

    provenance_counts.append(
        df[["temporal_grain_type", "date_assignment_method"]]
        .value_counts(dropna=False)
        .reset_index(name="count")
    )

    date_ranges.append({
        "file": path.name,
        "min_date": df["date"].min(),
        "max_date": df["date"].max(),
        "distinct_dates": df["date"].nunique(dropna=True),
        "rows": len(df),
    })

    del df

bus_temporal_qa_df = pd.DataFrame(qa_records)

bus_temporal_summary_df = (
    bus_temporal_qa_df
    .groupby("field", as_index=False)
    .agg(
        total_rows=("rows", "sum"),
        missing_count=("missing_count", "sum"),
        min_unique_values=("unique_values", "min"),
        max_unique_values=("unique_values", "max"),
    )
)

bus_temporal_summary_df["missing_rate"] = (
    bus_temporal_summary_df["missing_count"] /
    bus_temporal_summary_df["total_rows"]
)

display(bus_temporal_summary_df)

bus_date_range_df = pd.DataFrame(date_ranges)
print("Calendar-expanded Bus date coverage:")
display(bus_date_range_df)

bus_bucket_distribution_df = (
    pd.concat(bucket_counts, ignore_index=True)
    .groupby("temporal_bucket", dropna=False, as_index=False)["count"]
    .sum()
    .sort_values("temporal_bucket")
)

print("Temporal bucket distribution:")
display(bus_bucket_distribution_df)

bus_pre_post_distribution_df = (
    pd.concat(pre_post_counts, ignore_index=True)
    .groupby("pre_post_cp", dropna=False, as_index=False)["count"]
    .sum()
)

print("Pre/post congestion pricing distribution:")
display(bus_pre_post_distribution_df)

bus_provenance_distribution_df = (
    pd.concat(provenance_counts, ignore_index=True)
    .groupby(["temporal_grain_type", "date_assignment_method"], dropna=False, as_index=False)["count"]
    .sum()
)

print("Bus temporal provenance distribution:")
display(bus_provenance_distribution_df)

# Validate the January 5, 2025 policy boundary explicitly. Bus records are
# calendar-expanded from monthly day-of-week profiles, so exact dates should
# allow January 1-4 to remain pre-CP and January 5 onward to become post-CP.
bus_cp_boundary_frames = []
for path in bus_temporal_files:
    boundary_df = pd.read_parquet(path, columns=["date", "pre_post_cp"])
    boundary_df["date"] = pd.to_datetime(boundary_df["date"], errors="coerce")
    boundary_df = boundary_df[
        boundary_df["date"].between(
            pd.Timestamp("2025-01-01"),
            pd.Timestamp("2025-01-10"),
        )
    ]
    bus_cp_boundary_frames.append(boundary_df)

bus_cp_boundary_validation_df = pd.concat(
    bus_cp_boundary_frames,
    ignore_index=True,
)

bus_cp_boundary_validation_df = (
    bus_cp_boundary_validation_df
    .groupby(["date", "pre_post_cp"], as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
    .sort_values(["date", "pre_post_cp"])
)

print("January 2025 CP boundary validation:")
display(bus_cp_boundary_validation_df)

,field,total_rows,missing_count,min_unique_values,max_unique_values,missing_rate
0,date,82363372,0,402,731,0.0
1,date_assignment_method,82363372,0,1,1,0.0
2,day_of_week,82363372,0,7,7,0.0
3,daypart,82363372,0,5,5,0.0
4,daypart_order,82363372,0,5,5,0.0
5,hour,82363372,0,24,24,0.0
6,month,82363372,0,12,12,0.0
7,observation_sequence_id,82363372,0,2010,3655,0.0
8,pre_post_cp,82363372,0,1,2,0.0
9,temporal_bucket,82363372,0,10,10,0.0


Calendar-expanded Bus date coverage:


,file,min_date,max_date,distinct_dates,rows
0,bus_2023_2024_part_0000.parquet,2023-01-01,2024-12-31,710,2178646
1,bus_2023_2024_part_0001.parquet,2023-01-01,2024-12-31,731,2176192
2,bus_2023_2024_part_0002.parquet,2023-01-01,2024-12-31,731,2178484
3,bus_2023_2024_part_0003.parquet,2023-01-01,2024-12-31,731,2176069
4,bus_2023_2024_part_0004.parquet,2023-01-01,2024-12-31,731,2177873
5,bus_2023_2024_part_0005.parquet,2023-01-01,2024-12-31,731,2177889
6,bus_2023_2024_part_0006.parquet,2023-01-01,2024-12-31,731,2177031
7,bus_2023_2024_part_0007.parquet,2023-01-01,2024-12-31,731,2175608
8,bus_2023_2024_part_0008.parquet,2023-01-01,2024-12-31,731,2178740
9,bus_2023_2024_part_0009.parquet,2023-01-01,2024-12-31,731,2177089


Temporal bucket distribution:


,temporal_bucket,count
0,weekday_am_peak,12021475
1,weekday_evening,10178724
2,weekday_midday,17127625
3,weekday_overnight,9648782
4,weekday_pm_peak,12143560
5,weekend_am_peak,3751471
6,weekend_evening,3766181
7,weekend_midday,6179835
8,weekend_overnight,3449886
9,weekend_pm_peak,4095833


...

,pre_post_cp,count
0,post_cp,31314242
1,pre_cp,51049130


Bus temporal provenance distribution:


,temporal_grain_type,date_assignment_method,count
0,calendar_expanded_profile,expanded_from_year_month_day_of_week,82363372


January 2025 CP boundary validation:


,date,pre_post_cp,row_count
0,2025-01-01,pre_cp,76843
1,2025-01-02,pre_cp,74816
2,2025-01-03,pre_cp,74899
3,2025-01-04,pre_cp,65400
4,2025-01-05,post_cp,59677
5,2025-01-06,post_cp,72171
6,2025-01-07,post_cp,70715
7,2025-01-08,post_cp,76843
8,2025-01-09,post_cp,74816
9,2025-01-10,post_cp,74899


Findings\. Temporal standardization completed successfully for the bus dataset using calendar\-expanded profile dates\. The source Bus data remains a monthly Day\-of\-Week × Hour × Route Segment profile dataset, but this step now reconstructs all matching calendar dates within the study period so downstream tables can use exact \`date\`, exact \`pre\_post\_cp\`, and deterministic temporal ordering fields\. The temporal provenance columns preserve the fact that these are expanded profile dates rather than native daily observations\.

## 1\.2\.1\.3 Add temporal fields to subway data

The Subway activity dataset is already staged as quarterly parquet files from the source\-loading step\. In this section, we standardize its temporal representation so it can align cleanly with the rest of the mobility pipeline\. The authoritative time field is \`transit\_timestamp\`\. We preserve both validated Subway base metrics—\`ridership\` and \`transfers\`—while adding the shared project fields for date, calendar position, daypart, temporal bucket, ordering, and congestion\-pricing period\. The downstream analytical target remains:

\`Taxi Zone × Date × Temporal Bucket\`

At this stage, no Subway activity metric is aggregated or removed\. Ridership captures total station activity, while Transfers preserves the distinct transfer\-hub signal established in Notebook 1\.1\.5\.

### Test\-derive the temporal fields

Let's define the paths, constants, and helper logic needed to standardize Subway timestamps\. We keep the temporal bucket definitions identical to the rest of the project so Subway can later align cleanly with Traffic, Bus, and TLC at the shared \`Date × Temporal Bucket\` level\.

In [11]:
# ---------------------------------------------------------------------
# 1.2.1.3 Add temporal fields to subway data
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# Input / output paths
# ---------------------------------------------------------------------

SUBWAY_INPUT_DIR = Path("pipeline_data/1.1.5.subway_hourly_ridership_parquet")
SUBWAY_TEMPORAL_OUTPUT_DIR = Path("pipeline_data/1.2.1.subway_ridership_temporal_parquet")

SUBWAY_TEMPORAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBWAY_REQUIRED_TEMPORAL_COLUMNS = [
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
    "temporal_grain_type",
    "date_assignment_method",
]

SUBWAY_REQUIRED_METRIC_COLUMNS = [
    "ridership",
    "transfers",
]

SUBWAY_REQUIRED_OUTPUT_COLUMNS = (
    SUBWAY_REQUIRED_METRIC_COLUMNS
    + SUBWAY_REQUIRED_TEMPORAL_COLUMNS
)

Let's inspect the staged Subway parquet files before we begin transforming them\. We want to confirm the expected quarterly coverage is present and verify the timestamp field we'll use as the authoritative source for deriving the project's standardized temporal fields\.

In [12]:
# ---------------------------------------------------------------------
# Inspect staged subway parquet files
# ---------------------------------------------------------------------

subway_files = sorted(SUBWAY_INPUT_DIR.glob("*.parquet"))

print(f"Found {len(subway_files):,} parquet files:\n")

for file in subway_files:
    print(file.name)

Found 13 parquet files:

subway_2023_q1.parquet
subway_2023_q2.parquet
subway_2023_q3.parquet
subway_2023_q4.parquet
subway_2024_q1.parquet
subway_2024_q2.parquet
subway_2024_q3.parquet
subway_2024_q4.parquet
subway_2025_q1.parquet
subway_2025_q2.parquet
subway_2025_q3.parquet
subway_2025_q4.parquet
subway_2026_q1.parquet


Before deriving any temporal fields, let's remind ourselves of the schema of a representative Subway file\. This gives us one final sanity check on the timestamp field and confirms the structure matches what we validated during source staging\.

In [13]:
# ---------------------------------------------------------------------
# Inspect subway schema
# ---------------------------------------------------------------------

sample_file = subway_files[0]

schema_df = duckdb.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{sample_file}')
""").df()

schema_df

,column_name,column_type,null,key,default,extra
0,transit_timestamp,TIMESTAMP,YES,None,None,None
1,transit_mode,VARCHAR,YES,None,None,None
2,station_complex_id,VARCHAR,YES,None,None,None
3,station_complex,VARCHAR,YES,None,None,None
4,borough,VARCHAR,YES,None,None,None
5,payment_method,VARCHAR,YES,None,None,None
6,fare_class_category,VARCHAR,YES,None,None,None
7,ridership,DOUBLE,YES,None,None,None
8,transfers,DOUBLE,YES,None,None,None
9,latitude,DOUBLE,YES,None,None,None


Findings: The staged Subway schema looks clean and ready for temporal standardization\. Most importantly, \`transit\_timestamp\` is already stored as a proper \`TIMESTAMP\`, allowing us to derive the project's standard temporal fields directly without additional parsing\. The schema also confirms that ridership data is currently organized at the station\-complex level, with geographic attributes preserved for the spatial standardization step that follows\.

Let's use \`transit\_timestamp\` as the authoritative Subway timestamp and derive the standard temporal fields from it\. This keeps Subway aligned with the rest of the pipeline without adding any subway\-specific time logic\.

In [14]:
# ---------------------------------------------------------------------
# Create standardized temporal fields for one sample subway file
# ---------------------------------------------------------------------

sample_temporal_df = duckdb.sql(f"""
    SELECT
        *,
        CAST(transit_timestamp AS DATE) AS date,
        EXTRACT(year FROM transit_timestamp) AS year,
        EXTRACT(month FROM transit_timestamp) AS month,
        STRFTIME(transit_timestamp, '%A') AS day_of_week,
        EXTRACT(hour FROM transit_timestamp) AS hour
    FROM read_parquet('{sample_file}')
    LIMIT 10
""").df()

sample_temporal_df[
    [
        "transit_timestamp",
        "date",
        "year",
        "month",
        "day_of_week",
        "hour"
    ]
]

,transit_timestamp,date,year,month,day_of_week,hour
0,2023-01-01,2023-01-01,2023,1,Sunday,0
1,2023-01-01,2023-01-01,2023,1,Sunday,0
2,2023-01-01,2023-01-01,2023,1,Sunday,0
3,2023-01-01,2023-01-01,2023,1,Sunday,0
4,2023-01-01,2023-01-01,2023,1,Sunday,0
5,2023-01-01,2023-01-01,2023,1,Sunday,0
6,2023-01-01,2023-01-01,2023,1,Sunday,0
7,2023-01-01,2023-01-01,2023,1,Sunday,0
8,2023-01-01,2023-01-01,2023,1,Sunday,0
9,2023-01-01,2023-01-01,2023,1,Sunday,0


Findings: The temporal fields were derived successfully from \`transit\_timestamp\`\. The sample output shows the expected values for \`date\`, \`year\`, \`month\`, \`day\_of\_week\`, and \`hour\`, confirming that the staged timestamp can serve as the authoritative source for temporal standardization across the Subway dataset\.

Now that the basic timestamp fields look right, let's preview the two remaining standardized fields: \`temporal\_bucket\` and \`pre\_post\_cp\`\. This lets us validate the full temporal logic on a small sample before writing any new Subway parquet files\.

In [15]:
# ---------------------------------------------------------------------
# Preview full temporal-standardization logic on a sample
# ---------------------------------------------------------------------

sample_temporal_full_df = duckdb.sql(f"""
    SELECT
        transit_timestamp,
        CAST(transit_timestamp AS DATE) AS date,
        EXTRACT(year FROM transit_timestamp) AS year,
        EXTRACT(month FROM transit_timestamp) AS month,
        STRFTIME(transit_timestamp, '%A') AS day_of_week,
        EXTRACT(hour FROM transit_timestamp) AS hour,

        CASE
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 0 AND 5 THEN 'overnight'
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 6 AND 9 THEN 'am_peak'
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 10 AND 15 THEN 'midday'
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 16 AND 19 THEN 'pm_peak'
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 20 AND 23 THEN 'evening'
        END AS daypart,

        CASE
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 0 AND 5 THEN 1
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 6 AND 9 THEN 2
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 10 AND 15 THEN 3
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 16 AND 19 THEN 4
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 20 AND 23 THEN 5
        END AS daypart_order,

        CASE
            WHEN STRFTIME(transit_timestamp, '%A') IN ('Saturday', 'Sunday') THEN 'weekend_'
            ELSE 'weekday_'
        END ||
        CASE
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 0 AND 5 THEN 'overnight'
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 6 AND 9 THEN 'am_peak'
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 10 AND 15 THEN 'midday'
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 16 AND 19 THEN 'pm_peak'
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 20 AND 23 THEN 'evening'
        END AS temporal_bucket,

        CASE
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 0 AND 5 THEN 1
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 6 AND 9 THEN 2
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 10 AND 15 THEN 3
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 16 AND 19 THEN 4
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 20 AND 23 THEN 5
        END AS temporal_bucket_order,

        DATE_DIFF('day', DATE '2023-01-01', CAST(transit_timestamp AS DATE)) * 5 +
        CASE
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 0 AND 5 THEN 1
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 6 AND 9 THEN 2
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 10 AND 15 THEN 3
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 16 AND 19 THEN 4
            WHEN EXTRACT(hour FROM transit_timestamp) BETWEEN 20 AND 23 THEN 5
        END AS observation_sequence_id,

        CASE
            WHEN CAST(transit_timestamp AS DATE) < DATE '2025-01-05'
                THEN 'pre_cp'
            ELSE 'post_cp'
        END AS pre_post_cp

    FROM read_parquet('{sample_file}')
    WHERE CAST(transit_timestamp AS DATE)
        BETWEEN DATE '2023-01-01' AND DATE '2026-03-31'
    LIMIT 20
""").df()

sample_temporal_full_df

,transit_timestamp,date,year,month,day_of_week,hour,daypart,daypart_order,temporal_bucket,temporal_bucket_order,observation_sequence_id,pre_post_cp
0,2023-01-01,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp
1,2023-01-01,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp
2,2023-01-01,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp
3,2023-01-01,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp
4,2023-01-01,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp
5,2023-01-01,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp
6,2023-01-01,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp
7,2023-01-01,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp
8,2023-01-01,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp
9,2023-01-01,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp


Findings: The sample output confirms that the full temporal\-standardization logic is working as expected\. The derived fields correctly identify January 1, 2023 as a Sunday at hour 0, assign it to the \`weekend\_overnight\` temporal bucket, and classify it as occurring before the start of congestion pricing\. With the temporal logic validated, we can now generate the standardized Subway parquet files\.

### Write new temporal fields into parquet

Let's generate the temporal\-standardized Subway parquet files only when needed\. Since this is a heavier write step, we use a manual regeneration toggle and also check whether the expected output files already exist\. This lets the notebook safely skip the write when outputs are already present, while still generating them automatically the first time\.

In [16]:
# ---------------------------------------------------------------------
# Generate temporal-standardized subway files
# ---------------------------------------------------------------------

REGENERATE_SUBWAY_TEMPORAL_FILES = False

expected_output_files = [
    SUBWAY_TEMPORAL_OUTPUT_DIR / input_file.name
    for input_file in subway_files
]

existing_subway_temporal_files = sorted(SUBWAY_TEMPORAL_OUTPUT_DIR.glob("*.parquet"))
existing_subway_temporal_is_current = False

if existing_subway_temporal_files:
    sample_schema = pq.read_schema(existing_subway_temporal_files[0])
    existing_columns = set(sample_schema.names)
    existing_subway_temporal_is_current = set(SUBWAY_REQUIRED_OUTPUT_COLUMNS).issubset(
        existing_columns
    )

needs_subway_generation = (
    REGENERATE_SUBWAY_TEMPORAL_FILES
    or not all(path.exists() for path in expected_output_files)
    or not existing_subway_temporal_is_current
)

if needs_subway_generation:
    print("Generating temporal-standardized Subway parquet files...")

    for input_file, output_file in zip(subway_files, expected_output_files):
        print(f"Processing {input_file.name} -> {output_file.name}")

        if output_file.exists():
            output_file.unlink()

        duckdb.sql(f"""
            COPY (
                WITH base AS (
                    SELECT
                        *,
                        CAST(transit_timestamp AS DATE) AS date,
                        EXTRACT(year FROM transit_timestamp) AS year,
                        EXTRACT(month FROM transit_timestamp) AS month,
                        STRFTIME(transit_timestamp, '%A') AS day_of_week,
                        EXTRACT(hour FROM transit_timestamp) AS hour
                    FROM read_parquet('{input_file}')
                    WHERE CAST(transit_timestamp AS DATE)
                        BETWEEN DATE '2023-01-01' AND DATE '2026-03-31'
                ),

                ordered AS (
                    SELECT
                        *,
                        CASE
                            WHEN hour BETWEEN 0 AND 5 THEN 'overnight'
                            WHEN hour BETWEEN 6 AND 9 THEN 'am_peak'
                            WHEN hour BETWEEN 10 AND 15 THEN 'midday'
                            WHEN hour BETWEEN 16 AND 19 THEN 'pm_peak'
                            WHEN hour BETWEEN 20 AND 23 THEN 'evening'
                        END AS daypart,
                        CASE
                            WHEN hour BETWEEN 0 AND 5 THEN 1
                            WHEN hour BETWEEN 6 AND 9 THEN 2
                            WHEN hour BETWEEN 10 AND 15 THEN 3
                            WHEN hour BETWEEN 16 AND 19 THEN 4
                            WHEN hour BETWEEN 20 AND 23 THEN 5
                        END AS daypart_order
                    FROM base
                )

                SELECT
                    *,
                    CASE
                        WHEN day_of_week IN ('Saturday', 'Sunday') THEN 'weekend_'
                        ELSE 'weekday_'
                    END || daypart AS temporal_bucket,
                    daypart_order AS temporal_bucket_order,
                    DATE_DIFF('day', DATE '2023-01-01', date) * 5
                        + daypart_order AS observation_sequence_id,
                    CASE
                        WHEN date < DATE '2025-01-05' THEN 'pre_cp'
                        ELSE 'post_cp'
                    END AS pre_post_cp,
                    'native_timestamp' AS temporal_grain_type,
                    'derived_from_native_timestamp' AS date_assignment_method
                FROM ordered
            )
            TO '{output_file}'
            (FORMAT PARQUET)
        """)

    print("\nTemporal-standardized Subway parquet files are ready.")

else:
    print("\nTemporal-standardized Subway parquet files already exist with the current schema. Skipping generation.")


Temporal-standardized Subway parquet files already exist with the current schema. Skipping generation.


### Subway Parquet Validation

Now that the temporal\-standardized Subway files have been written, let's validate only the transformation\-level checks that matter here: row preservation, temporal bucket coverage, and congestion\-pricing period coverage\. The deeper source\-data integrity checks were already handled during Subway staging\.

In [17]:
# ---------------------------------------------------------------------
# Validate row and Subway metric preservation after study-period filtering
# ---------------------------------------------------------------------

subway_input_glob = str(SUBWAY_INPUT_DIR / "*.parquet")
subway_temporal_output_glob = str(
    SUBWAY_TEMPORAL_OUTPUT_DIR / "*.parquet"
)

subway_row_preservation_df = duckdb.sql(f"""
    WITH input_summary AS (
        SELECT
            COUNT(*) AS input_row_count_in_study_period,
            SUM(ridership) AS input_ridership,
            SUM(transfers) AS input_transfers
        FROM read_parquet('{subway_input_glob}')
        WHERE CAST(transit_timestamp AS DATE)
            BETWEEN DATE '2023-01-01' AND DATE '2026-03-31'
    ),

    output_summary AS (
        SELECT
            COUNT(*) AS output_row_count,
            SUM(ridership) AS output_ridership,
            SUM(transfers) AS output_transfers
        FROM read_parquet('{subway_temporal_output_glob}')
    )

    SELECT
        input_row_count_in_study_period,
        output_row_count,
        output_row_count
            - input_row_count_in_study_period
            AS row_count_difference,

        input_ridership,
        output_ridership,
        output_ridership
            - input_ridership
            AS ridership_difference,

        input_transfers,
        output_transfers,
        output_transfers
            - input_transfers
            AS transfers_difference

    FROM input_summary
    CROSS JOIN output_summary
""").df()

subway_row_preservation_df

,input_row_count_in_study_period,output_row_count,row_count_difference,input_ridership,output_ridership,ridership_difference,input_transfers,output_transfers,transfers_difference
0,88319707,88319707,0,3.985438e+09,3.985438e+09,0.0,176532431.0,176532431.0,0.0


Findings\. The temporal\-standardized Subway dataset passed the full preservation check after applying the project study\-period filter\. All 88,319,707 in\-period observations were retained, with no change to the cumulative Ridership total of 3,985,437,979 or the cumulative Transfers total of 176,532,431\. The output is therefore constrained to January 1, 2023 through March 31, 2026 while preserving both Subway base metrics and adding the project\-wide daypart, temporal ordering, congestion\-pricing, and temporal provenance fields required for downstream harmonization\.


Let's confirm that the new temporal bucket field covers the full standardized bucket framework\. This checks the main categorization logic we added in this step\.

In [18]:
# ---------------------------------------------------------------------
# Validate temporal contract fields and bucket coverage
# ---------------------------------------------------------------------

subway_temporal_contract_validation_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        SUM(CASE WHEN date < DATE '2023-01-01' OR date > DATE '2026-03-31' THEN 1 ELSE 0 END) AS out_of_range_rows,
        SUM(CASE WHEN hour NOT BETWEEN 0 AND 23 THEN 1 ELSE 0 END) AS invalid_hour_rows,
        SUM(CASE WHEN daypart IS NULL THEN 1 ELSE 0 END) AS missing_daypart_rows,
        SUM(CASE WHEN daypart_order IS NULL THEN 1 ELSE 0 END) AS missing_daypart_order_rows,
        SUM(CASE WHEN temporal_bucket_order IS NULL THEN 1 ELSE 0 END) AS missing_bucket_order_rows,
        SUM(CASE WHEN observation_sequence_id IS NULL THEN 1 ELSE 0 END) AS missing_sequence_rows,
        SUM(CASE WHEN pre_post_cp NOT IN ('pre_cp', 'post_cp') THEN 1 ELSE 0 END) AS unexpected_cp_rows
    FROM read_parquet('{subway_temporal_output_glob}')
""").df()

display(subway_temporal_contract_validation_df)

subway_temporal_bucket_validation_df = duckdb.sql(f"""
    SELECT
        temporal_bucket,
        MIN(temporal_bucket_order) AS temporal_bucket_order,
        COUNT(*) AS row_count
    FROM read_parquet('{subway_temporal_output_glob}')
    GROUP BY temporal_bucket
    ORDER BY temporal_bucket_order, temporal_bucket
""").df()

subway_temporal_bucket_validation_df

,row_count,min_date,max_date,out_of_range_rows,invalid_hour_rows,missing_daypart_rows,missing_daypart_order_rows,missing_bucket_order_rows,missing_sequence_rows,unexpected_cp_rows
0,88319707,2023-01-01,2026-03-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,temporal_bucket,temporal_bucket_order,row_count
0,weekday_overnight,1,10831106
1,weekend_overnight,1,4044099
2,weekday_am_peak,2,12600063
3,weekend_am_peak,2,4169571
4,weekday_midday,3,18990788
5,weekend_midday,3,6706761
6,weekday_pm_peak,4,12470066
7,weekend_pm_peak,4,4354727
8,weekday_evening,5,10323543
9,weekend_evening,5,3828983


Findings: All ten expected temporal buckets are present in the standardized Subway dataset, confirming complete coverage of the project's shared temporal framework\. The distribution of observations appears reasonable, with weekday buckets containing substantially more ridership observations than weekend buckets and midday periods containing the highest overall activity levels\. No unexpected bucket values were observed, indicating that all Subway records were assigned successfully to a valid temporal bucket\.

Let's also confirm that the new congestion\-pricing period field splits observations into the expected pre\- and post\-policy periods\.

In [19]:
# ---------------------------------------------------------------------
# Validate congestion-pricing period coverage
# ---------------------------------------------------------------------

subway_pre_post_validation_df = duckdb.sql(f"""
    SELECT
        pre_post_cp,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(*) AS row_count
    FROM read_parquet('{subway_temporal_output_glob}')
    GROUP BY pre_post_cp
    ORDER BY pre_post_cp
""").df()

subway_pre_post_validation_df

,pre_post_cp,min_date,max_date,row_count
0,post_cp,2025-01-05,2026-03-30,35417907
1,pre_cp,2023-01-01,2025-01-04,52901800


Findings: The congestion\-pricing assignments appear correct and cover the full study period\. The pre\-congestion\-pricing period spans January 1, 2023 through January 4, 2025, while the post\-congestion\-pricing period begins on January 5, 2025\. The policy boundary aligns exactly with the project's congestion\-pricing start date, and the standardized labels now use the project\-wide \`pre\_cp\` / \`post\_cp\` values\.

### Section Summary

Subway temporal standardization is complete\. Standardized temporal fields were derived successfully from \`transit\_timestamp\`, all 88\.3 million observations were preserved, all ten temporal buckets were populated, and congestion\-pricing assignments aligned correctly with the project's policy boundary\. The resulting dataset is now ready for Taxi Zone assignment and spatial standardization in Section 1\.2\.2\.

## 1\.2\.1\.4 Add temporal fields to bridge & tunnel

The Bridges & Tunnels dataset is already reported at an hourly level, but it still needs to be aligned with the project's standard temporal schema before it can be used in downstream multimodal analysis\. In this section, we load the staged Bridges & Tunnels parquet file, convert the source timestamp and date fields into consistent datetime formats, and add the shared temporal fields used across the rest of the pipeline, including year, month, day of week, temporal bucket, and congestion\-pricing period\.

### Load Staged Bridges & Tunnels Dataset

First, define the input and output paths for the Bridges & Tunnels temporal standardization step\. The input is the staged parquet created during the loading notebook, while the output will be the temporally standardized parquet used in later pipeline stages\.

In [20]:
# ---------------------------------------------------------------------
# 1.2.1.4 Add temporal fields to bridge & tunnel
# ---------------------------------------------------------------------

REBUILD_BRIDGE_TUNNEL_TEMPORAL = False

BRIDGE_TUNNEL_STAGED_PARQUET_PATH = (
    PIPELINE_DATA_DIR /
    "1.1.3.bridge_tunnel_hourly_crossings" /
    "bridge_tunnel_hourly_crossings_staged.parquet"
)

BRIDGE_TUNNEL_TEMPORAL_PARQUET_PATH = (
    PIPELINE_DATA_DIR /
    "1.2.1.bridge_tunnel_temporal.parquet"
)

Next, load the staged Bridges & Tunnels parquet file and confirm that the dataset is available for temporal standardization\.

In [21]:
bridge_tunnel_temporal = pd.read_parquet(
    BRIDGE_TUNNEL_STAGED_PARQUET_PATH
)

print("Bridge & Tunnel staged shape:", bridge_tunnel_temporal.shape)

display(bridge_tunnel_temporal.head())

Bridge & Tunnel staged shape: (5941375, 11)


,transit_timestamp,date,hour,facility_id,facility,direction,payment_method,vehicle_class,vehicle_class_description,vehicle_class_category,traffic_count
0,01/01/2023 12:00:00 AM,01/01/2023,0,27,Queens Midtown Tunnel,Westbound to Manhattan,Tolls by Mail,9,motorcycle,Motorcycle,2
1,01/01/2023 12:00:00 AM,01/01/2023,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,Tolls by Mail,1,2-axle passenger car,Car,83
2,01/01/2023 12:00:00 AM,01/01/2023,0,29,Throgs Neck Bridge,Southbound to Queens,E-ZPass,6,3-axle truck,Truck,1
3,01/01/2023 12:00:00 AM,01/01/2023,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,E-ZPass,1,2-axle passenger car,Car,404
4,01/01/2023 12:00:00 AM,01/01/2023,0,30,Verrazzano - Narrows Bridge,Eastbound to Brooklyn,E-ZPass,8,5-axle truck,Truck,3


Finally, inspect the staged schema before adding new temporal fields\. This helps confirm that the expected timestamp, date, hour, facility, vehicle classification, payment method, and traffic count fields are present before transformation\.

In [22]:
bridge_tunnel_temporal.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5941375 entries, 0 to 5941374
Data columns (total 11 columns):
 #   Column                     Dtype 
---  ------                     ----- 
 0   transit_timestamp          object
 1   date                       object
 2   hour                       int64 
 3   facility_id                int64 
 4   facility                   object
 5   direction                  object
 6   payment_method             object
 7   vehicle_class              int64 
 8   vehicle_class_description  object
 9   vehicle_class_category     object
 10  traffic_count              int64 
dtypes: int64(4), object(7)
memory usage: 498.6+ MB


### Generate Standard Temporal Features

Next, we add the standard temporal fields used across the congestion\-pricing pipeline\. Although the Bridges & Tunnels dataset is already reported at an hourly level, we still need to normalize the timestamp and date fields and generate the shared temporal dimensions used by the other mobility datasets\. This includes year, month, day of week, weekday/weekend daypart buckets, and the pre/post congestion\-pricing indicator\.

First, convert the source timestamp and date fields into proper datetime objects\. This ensures that downstream temporal fields are derived from consistent datetime values rather than string representations\.

In [23]:
bridge_tunnel_temporal["transit_timestamp"] = pd.to_datetime(
    bridge_tunnel_temporal["transit_timestamp"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

bridge_tunnel_temporal["date"] = pd.to_datetime(
    bridge_tunnel_temporal["date"],
    format="%m/%d/%Y",
    errors="coerce"
).dt.normalize()

bridge_tunnel_temporal = filter_to_study_period(
    bridge_tunnel_temporal,
    date_col="date"
)

Next, derive the standard calendar fields used throughout the project\. These fields make the Bridges & Tunnels dataset consistent with the temporal schema already applied to Traffic, Bus, Subway, and TLC data\.

In [24]:
bridge_tunnel_temporal["year"] = pd.to_datetime(
    bridge_tunnel_temporal["date"]
).dt.year

bridge_tunnel_temporal["month"] = pd.to_datetime(
    bridge_tunnel_temporal["date"]
).dt.month

bridge_tunnel_temporal["day_of_week"] = pd.to_datetime(
    bridge_tunnel_temporal["date"]
).dt.day_name()

bridge_tunnel_temporal["hour"] = bridge_tunnel_temporal["hour"].astype(int)

display(bridge_tunnel_temporal.head())

,transit_timestamp,date,hour,facility_id,facility,direction,payment_method,vehicle_class,vehicle_class_description,vehicle_class_category,traffic_count,year,month,day_of_week
0,2023-01-01,2023-01-01,0,27,Queens Midtown Tunnel,Westbound to Manhattan,Tolls by Mail,9,motorcycle,Motorcycle,2,2023,1,Sunday
1,2023-01-01,2023-01-01,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,Tolls by Mail,1,2-axle passenger car,Car,83,2023,1,Sunday
2,2023-01-01,2023-01-01,0,29,Throgs Neck Bridge,Southbound to Queens,E-ZPass,6,3-axle truck,Truck,1,2023,1,Sunday
3,2023-01-01,2023-01-01,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,E-ZPass,1,2-axle passenger car,Car,404,2023,1,Sunday
4,2023-01-01,2023-01-01,0,30,Verrazzano - Narrows Bridge,Eastbound to Brooklyn,E-ZPass,8,5-axle truck,Truck,3,2023,1,Sunday


Then, assign each hourly observation to the project's standard temporal bucket definitions\. These buckets combine weekday/weekend behavior with daypart structure so that each mobility dataset can be compared on the same time\-of\-week basis\.

In [25]:
bridge_tunnel_temporal["temporal_bucket"] = bridge_tunnel_temporal.apply(
    lambda row: assign_temporal_bucket(
        row["day_of_week"],
        row["hour"]
    ),
    axis=1
)

bridge_tunnel_temporal = add_temporal_ordering_fields(
    bridge_tunnel_temporal,
    date_col="date"
)

bridge_tunnel_temporal[[
    "date",
    "day_of_week",
    "hour",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
]].head()

,date,day_of_week,hour,daypart,daypart_order,temporal_bucket,temporal_bucket_order,observation_sequence_id
0,2023-01-01,Sunday,0,overnight,1,weekend_overnight,1,1
1,2023-01-01,Sunday,0,overnight,1,weekend_overnight,1,1
2,2023-01-01,Sunday,0,overnight,1,weekend_overnight,1,1
3,2023-01-01,Sunday,0,overnight,1,weekend_overnight,1,1
4,2023-01-01,Sunday,0,overnight,1,weekend_overnight,1,1


Finally, add the pre/post congestion\-pricing indicator using the same project cutoff date applied to the other mobility datasets\. This creates the core treatment\-period flag needed for downstream comparison and modeling\.

In [26]:
bridge_tunnel_temporal["pre_post_cp"] = assign_pre_post_cp(
    bridge_tunnel_temporal["date"]
)
bridge_tunnel_temporal["date"] = pd.to_datetime(bridge_tunnel_temporal["date"]).dt.date
bridge_tunnel_temporal["temporal_grain_type"] = "native_hourly_timestamp"
bridge_tunnel_temporal["date_assignment_method"] = "derived_from_native_facility_timestamp"

validate_temporal_contract(bridge_tunnel_temporal, "Bridge & Tunnel")

bridge_tunnel_temporal[[
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
    "temporal_grain_type",
    "date_assignment_method",
]].head()

Bridge & Tunnel temporal contract checks passed.


,date,year,month,day_of_week,hour,daypart,daypart_order,temporal_bucket,temporal_bucket_order,observation_sequence_id,pre_post_cp,temporal_grain_type,date_assignment_method
0,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp,native_hourly_timestamp,derived_from_native_facility_timestamp
1,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp,native_hourly_timestamp,derived_from_native_facility_timestamp
2,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp,native_hourly_timestamp,derived_from_native_facility_timestamp
3,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp,native_hourly_timestamp,derived_from_native_facility_timestamp
4,2023-01-01,2023,1,Sunday,0,overnight,1,weekend_overnight,1,1,pre_cp,native_hourly_timestamp,derived_from_native_facility_timestamp


The standard temporal fields were successfully added to the Bridges & Tunnels dataset, bringing it into alignment with the temporal schema used across the other mobility sources\. The resulting dataset now contains consistent calendar attributes, temporal buckets, and congestion\-pricing period indicators that will support downstream integration, aggregation, and comparative analysis across modalities\.

### Validate Temporal Coverage

Let's now validate their coverage and completeness\. This QA step confirms that all records were successfully assigned the expected temporal dimensions and that the resulting values align with the dataset's observed date range and reporting structure\. Consistent temporal coverage is essential for ensuring that the Bridges & Tunnels dataset can be integrated reliably with the project's other mobility sources\.

First, check whether any of the standardized temporal fields contain missing values\. This confirms that timestamp parsing, calendar feature generation, temporal bucket assignment, and congestion\-pricing period assignment completed successfully for every record\.

In [27]:
# ---------------------------------------------------------------------
# Validate Temporal Coverage
# ---------------------------------------------------------------------

bridge_tunnel_temporal_fields = [
    "transit_timestamp",
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
    "temporal_grain_type",
    "date_assignment_method",
]

bridge_tunnel_temporal_missingness_df = (
    bridge_tunnel_temporal[bridge_tunnel_temporal_fields]
    .isna()
    .sum()
    .reset_index()
    .rename(columns={
        "index": "field",
        0: "missing_count"
    })
)

bridge_tunnel_temporal_missingness_df["missing_pct"] = (
    bridge_tunnel_temporal_missingness_df["missing_count"] /
    len(bridge_tunnel_temporal)
)

display(bridge_tunnel_temporal_missingness_df)

,field,missing_count,missing_pct
0,transit_timestamp,0,0.0
1,date,0,0.0
2,year,0,0.0
3,month,0,0.0
4,day_of_week,0,0.0
5,hour,0,0.0
6,daypart,0,0.0
7,daypart_order,0,0.0
8,temporal_bucket,0,0.0
9,temporal_bucket_order,0,0.0


Next, summarize the range and cardinality of the standardized temporal fields\. This gives us a compact view of the dataset's date range, hourly coverage, temporal bucket coverage, and pre/post congestion\-pricing coverage\.

In [28]:
bridge_tunnel_temporal_coverage_df = pd.DataFrame({
    "field": bridge_tunnel_temporal_fields,
    "min_value": [bridge_tunnel_temporal[col].min() for col in bridge_tunnel_temporal_fields],
    "max_value": [bridge_tunnel_temporal[col].max() for col in bridge_tunnel_temporal_fields],
    "distinct_values": [bridge_tunnel_temporal[col].nunique(dropna=True) for col in bridge_tunnel_temporal_fields],
})

display(bridge_tunnel_temporal_coverage_df)

,field,min_value,max_value,distinct_values
0,transit_timestamp,2023-01-01 00:00:00,2026-03-31 23:00:00,28460
1,date,2023-01-01,2026-03-31,1186
2,year,2023,2026,4
3,month,1,12,12
4,day_of_week,Friday,Wednesday,7
5,hour,0,23,24
6,daypart,am_peak,pm_peak,5
7,daypart_order,1,5,5
8,temporal_bucket,weekday_am_peak,weekend_pm_peak,10
9,temporal_bucket_order,1,5,5


Finally, review the distribution of records across the derived temporal buckets and congestion\-pricing periods\. These counts are based on source observations rather than traffic volumes, so they are mainly used to verify that the new fields were assigned as expected\.

In [29]:
bridge_tunnel_temporal_bucket_counts_df = (
    bridge_tunnel_temporal["temporal_bucket"]
    .value_counts()
    .rename_axis("temporal_bucket")
    .reset_index(name="observation_count")
)

display(bridge_tunnel_temporal_bucket_counts_df)

,temporal_bucket,observation_count
0,weekday_midday,1178848
1,weekday_overnight,886985
2,weekday_am_peak,785182
3,weekday_pm_peak,730826
4,weekday_evening,664094
5,weekend_midday,406818
6,weekend_overnight,325755
7,weekend_pm_peak,261022
8,weekend_am_peak,257481
9,weekend_evening,242860


In [30]:
bridge_tunnel_pre_post_counts_df = (
    bridge_tunnel_temporal["pre_post_cp"]
    .value_counts()
    .rename_axis("pre_post_cp")
    .reset_index(name="observation_count")
)

display(bridge_tunnel_pre_post_counts_df)

,pre_post_cp,observation_count
0,pre_cp,3602730
1,post_cp,2137141


Findings: All standardized temporal fields exhibited complete coverage with no missing values detected\. The dataset is filtered to the project study period, January 1, 2023 through March 31, 2026, and contains the expected range of years, months, days of week, hours, temporal buckets, ordering fields, and congestion\-pricing periods\. All temporal buckets and both pre\- and post\-congestion\-pricing periods were populated successfully, indicating that temporal standardization was applied consistently across the study\-period Bridge & Tunnel records\.

### Save Output

With the standard temporal fields added and validated, the final step is to save the Bridges & Tunnels dataset as a temporally standardized parquet file\. This output will serve as the canonical Bridge & Tunnel input for later harmonization and modeling steps, where it can be used as a global mobility signal joined by date and temporal bucket\.

First, write the temporally standardized Bridges & Tunnels dataset to the pipeline output directory\.

In [31]:
# ---------------------------------------------------------------------
# Save Output
# ---------------------------------------------------------------------

bridge_tunnel_temporal.to_parquet(
    BRIDGE_TUNNEL_TEMPORAL_PARQUET_PATH,
    index=False
)

print(
    f"Saved temporally standardized Bridges & Tunnels dataset to:\n"
    f"{BRIDGE_TUNNEL_TEMPORAL_PARQUET_PATH}"
)

Saved temporally standardized Bridges & Tunnels dataset to:
pipeline_data/1.2.1.bridge_tunnel_temporal.parquet


Next, reload the saved parquet file and confirm that the output preserves the expected row and column structure\.

In [32]:
bridge_tunnel_temporal_check = pd.read_parquet(
    BRIDGE_TUNNEL_TEMPORAL_PARQUET_PATH
)

print("Original shape:", bridge_tunnel_temporal.shape)
print("Reloaded shape:", bridge_tunnel_temporal_check.shape)

display(bridge_tunnel_temporal_check.head())

Original shape: (5739871, 22)
Reloaded shape: (5739871, 22)


,transit_timestamp,date,hour,facility_id,facility,direction,payment_method,vehicle_class,vehicle_class_description,vehicle_class_category,...,month,day_of_week,temporal_bucket,daypart,daypart_order,temporal_bucket_order,observation_sequence_id,pre_post_cp,temporal_grain_type,date_assignment_method
0,2023-01-01,2023-01-01,0,27,Queens Midtown Tunnel,Westbound to Manhattan,Tolls by Mail,9,motorcycle,Motorcycle,...,1,Sunday,weekend_overnight,overnight,1,1,1,pre_cp,native_hourly_timestamp,derived_from_native_facility_timestamp
1,2023-01-01,2023-01-01,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,Tolls by Mail,1,2-axle passenger car,Car,...,1,Sunday,weekend_overnight,overnight,1,1,1,pre_cp,native_hourly_timestamp,derived_from_native_facility_timestamp
2,2023-01-01,2023-01-01,0,29,Throgs Neck Bridge,Southbound to Queens,E-ZPass,6,3-axle truck,Truck,...,1,Sunday,weekend_overnight,overnight,1,1,1,pre_cp,native_hourly_timestamp,derived_from_native_facility_timestamp
3,2023-01-01,2023-01-01,0,28,Hugh L. Carey Tunnel,Northbound to Manhattan,E-ZPass,1,2-axle passenger car,Car,...,1,Sunday,weekend_overnight,overnight,1,1,1,pre_cp,native_hourly_timestamp,derived_from_native_facility_timestamp
4,2023-01-01,2023-01-01,0,30,Verrazzano - Narrows Bridge,Eastbound to Brooklyn,E-ZPass,8,5-axle truck,Truck,...,1,Sunday,weekend_overnight,overnight,1,1,1,pre_cp,native_hourly_timestamp,derived_from_native_facility_timestamp


Finally, compare the original and reloaded row counts to confirm that no records were dropped during the save process\.

In [33]:
row_count_difference = (
    len(bridge_tunnel_temporal) -
    len(bridge_tunnel_temporal_check)
)

print(f"Row count difference: {row_count_difference:,}")

Row count difference: 0


The temporally standardized Bridges & Tunnels dataset was successfully written to parquet and reloaded without issue\. Row counts matched exactly between the source and output datasets, indicating that no records were lost during the save process\. The resulting dataset preserves the full set of bridge and tunnel crossing observations while providing a consistent temporal schema that can be used alongside the project's other mobility datasets\.

## 1\.2\.1\.5 Final Temporal Contract Validation

This final checkpoint validates the temporal\-standardized outputs written by the notebook before downstream spatial harmonization begins\. It confirms that each output contains the project\-wide temporal contract fields, uses the approved study period and congestion\-pricing labels, preserves deterministic within\-day ordering fields, and carries temporal provenance metadata\. If this cell fails, downstream notebooks should not be run until the missing or invalid contract fields are repaired\.

In [34]:
# ---------------------------------------------------------------------
# Final Temporal Contract Validation
# ---------------------------------------------------------------------

FINAL_TEMPORAL_REQUIRED_COLUMNS = [
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
    "temporal_grain_type",
    "date_assignment_method",
]

FINAL_TEMPORAL_OUTPUTS = [
    {
        "dataset": "traffic_counts",
        "source": str(TRAFFIC_COUNTS_TEMPORAL_PATH),
        "column_aliases": {},
    },
    {
        "dataset": "bus_speeds",
        "source": str(BUS_TEMPORAL_DIR / "*.parquet"),
        "column_aliases": {
            "year": "Year",
            "month": "Month",
        },
    },
    {
        "dataset": "subway_ridership",
        "source": str(SUBWAY_TEMPORAL_OUTPUT_DIR / "*.parquet"),
        "column_aliases": {},
    },
    {
        "dataset": "bridge_tunnel",
        "source": str(BRIDGE_TUNNEL_TEMPORAL_PARQUET_PATH),
        "column_aliases": {},
    },
]

VALID_BUCKET_SQL = "(" + ", ".join(f"'{bucket}'" for bucket in sorted(VALID_TEMPORAL_BUCKETS)) + ")"
VALID_CP_SQL = "('pre_cp', 'post_cp')"

final_validation_records = []
schema_records = []

for output in FINAL_TEMPORAL_OUTPUTS:
    dataset = output["dataset"]
    source = output["source"]
    column_aliases = output.get("column_aliases", {})

    schema_df = duckdb.sql(f"""
        DESCRIBE
        SELECT *
        FROM read_parquet('{source}')
        LIMIT 0
    """).df()

    discovered_columns = list(schema_df["column_name"])
    discovered_column_set = set(discovered_columns)

    schema_records.append({
        "dataset": dataset,
        "source": source,
        "discovered_columns": discovered_columns,
    })

    resolved_columns = {}
    missing_columns = []
    alias_columns_used = []

    for required_col in FINAL_TEMPORAL_REQUIRED_COLUMNS:
        if required_col in discovered_column_set:
            resolved_columns[required_col] = required_col
        elif required_col in column_aliases and column_aliases[required_col] in discovered_column_set:
            resolved_columns[required_col] = column_aliases[required_col]
            alias_columns_used.append(f"{required_col}->{column_aliases[required_col]}")
        else:
            missing_columns.append(required_col)

    if missing_columns:
        schema_preview_df = pd.DataFrame(schema_records)
        display(schema_preview_df)
        raise ValueError(
            f"{dataset} missing required temporal columns: {missing_columns}. "
            f"Discovered columns: {discovered_columns}"
        )

    date_col = resolved_columns["date"]
    year_col = resolved_columns["year"]
    month_col = resolved_columns["month"]
    day_of_week_col = resolved_columns["day_of_week"]
    hour_col = resolved_columns["hour"]
    daypart_col = resolved_columns["daypart"]
    daypart_order_col = resolved_columns["daypart_order"]
    temporal_bucket_col = resolved_columns["temporal_bucket"]
    temporal_bucket_order_col = resolved_columns["temporal_bucket_order"]
    observation_sequence_col = resolved_columns["observation_sequence_id"]
    pre_post_cp_col = resolved_columns["pre_post_cp"]
    temporal_grain_type_col = resolved_columns["temporal_grain_type"]
    date_assignment_method_col = resolved_columns["date_assignment_method"]

    validation_df = duckdb.sql(f"""
        SELECT
            COUNT(*) AS row_count,
            MIN(CAST("{date_col}" AS DATE)) AS min_date,
            MAX(CAST("{date_col}" AS DATE)) AS max_date,
            COUNT(DISTINCT "{year_col}") AS distinct_years,
            COUNT(DISTINCT "{month_col}") AS distinct_months,
            SUM(CASE WHEN "{date_col}" IS NULL THEN 1 ELSE 0 END) AS missing_date_rows,
            SUM(CASE WHEN CAST("{date_col}" AS DATE) < DATE '2023-01-01'
                      OR CAST("{date_col}" AS DATE) > DATE '2026-03-31'
                     THEN 1 ELSE 0 END) AS out_of_study_period_rows,
            SUM(CASE WHEN "{hour_col}" IS NULL OR "{hour_col}" NOT BETWEEN 0 AND 23
                     THEN 1 ELSE 0 END) AS invalid_hour_rows,
            SUM(CASE WHEN "{day_of_week_col}" IS NULL THEN 1 ELSE 0 END) AS missing_day_of_week_rows,
            SUM(CASE WHEN "{daypart_col}" IS NULL THEN 1 ELSE 0 END) AS missing_daypart_rows,
            SUM(CASE WHEN "{daypart_order_col}" IS NULL THEN 1 ELSE 0 END) AS missing_daypart_order_rows,
            SUM(CASE WHEN "{temporal_bucket_col}" IS NULL
                      OR "{temporal_bucket_col}" NOT IN {VALID_BUCKET_SQL}
                     THEN 1 ELSE 0 END) AS invalid_temporal_bucket_rows,
            SUM(CASE WHEN "{temporal_bucket_order_col}" IS NULL THEN 1 ELSE 0 END) AS missing_bucket_order_rows,
            SUM(CASE WHEN "{observation_sequence_col}" IS NULL THEN 1 ELSE 0 END) AS missing_sequence_rows,
            SUM(CASE WHEN "{pre_post_cp_col}" IS NULL
                      OR "{pre_post_cp_col}" NOT IN {VALID_CP_SQL}
                     THEN 1 ELSE 0 END) AS invalid_pre_post_cp_rows,
            SUM(CASE WHEN "{temporal_grain_type_col}" IS NULL THEN 1 ELSE 0 END) AS missing_grain_type_rows,
            SUM(CASE WHEN "{date_assignment_method_col}" IS NULL THEN 1 ELSE 0 END) AS missing_date_assignment_rows
        FROM read_parquet('{source}')
    """).df()

    validation_record = validation_df.iloc[0].to_dict()
    validation_record["dataset"] = dataset
    validation_record["source"] = source
    validation_record["missing_columns"] = missing_columns
    validation_record["alias_columns_used"] = ", ".join(alias_columns_used)
    final_validation_records.append(validation_record)

final_temporal_contract_validation_df = pd.DataFrame(final_validation_records)

failure_columns = [
    "missing_date_rows",
    "out_of_study_period_rows",
    "invalid_hour_rows",
    "missing_day_of_week_rows",
    "missing_daypart_rows",
    "missing_daypart_order_rows",
    "invalid_temporal_bucket_rows",
    "missing_bucket_order_rows",
    "missing_sequence_rows",
    "invalid_pre_post_cp_rows",
    "missing_grain_type_rows",
    "missing_date_assignment_rows",
]

failed_checks = final_temporal_contract_validation_df[
    final_temporal_contract_validation_df[failure_columns].sum(axis=1) > 0
]

if not failed_checks.empty:
    display(final_temporal_contract_validation_df)
    raise ValueError("One or more final temporal contract checks failed.")

print("All final temporal contract checks passed.")
display(final_temporal_contract_validation_df)

All final temporal contract checks passed.


,row_count,min_date,max_date,distinct_years,distinct_months,missing_date_rows,out_of_study_period_rows,invalid_hour_rows,missing_day_of_week_rows,missing_daypart_rows,...,invalid_temporal_bucket_rows,missing_bucket_order_rows,missing_sequence_rows,invalid_pre_post_cp_rows,missing_grain_type_rows,missing_date_assignment_rows,dataset,source,missing_columns,alias_columns_used
0,266651,2023-01-06,2026-02-03,4,12,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,traffic_counts,pipeline_data/1.2.1.traffic_counts_temporal.pa...,[],
1,82363372,2023-01-01,2026-03-31,4,12,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,bus_speeds,pipeline_data/1.2.1.bus_speeds_temporal_parque...,[],"year->Year, month->Month"
2,88319707,2023-01-01,2026-03-30,4,12,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,subway_ridership,pipeline_data/1.2.1.subway_ridership_temporal_...,[],
3,5739871,2023-01-01,2026-03-31,4,12,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,bridge_tunnel,pipeline_data/1.2.1.bridge_tunnel_temporal.par...,[],


Findings\. Final temporal contract validation passed across all four temporally standardized outputs\. Traffic, Bus, Subway, and Bridge & Tunnel all contain the required date, calendar, daypart, temporal bucket, ordering, congestion\-pricing, and provenance fields with no missing values in the contract columns, no invalid hours, no unexpected temporal buckets, no invalid \`pre\_post\_cp\` values, and no observations outside the January 1, 2023 through March 31, 2026 study period\. Traffic remains a sampled roadway dataset with 266,651 observations spanning January 6, 2023 through February 3, 2026\. Bus now contains 82\.4 million calendar\-expanded profile observations covering the full study period, which confirms that the profile\-to\-calendar expansion worked and that the January 2025 policy boundary can be handled exactly\. Subway contains 88\.3 million hourly station\-complex observations through March 30, 2026, and Bridge & Tunnel contains 5\.7 million system\-level hourly crossing observations through March 31, 2026\. The only schema wrinkle is Bus retaining source\-style \`Year\` and \`Month\` column names while also passing validation through the \`year\-\>Year\` and \`month\-\>Month\` aliases; downstream harmonization should either normalize those names or explicitly preserve the alias policy\.

## Close

In this notebook, we standardized temporal representations across Traffic, Bus, Subway, and Bridge & Tunnel mobility datasets by deriving consistent date and time fields, assigning observations to the project's shared weekday/weekend daypart buckets, creating deterministic ordering fields, filtering canonical outputs to the study period, and assigning \`pre\_cp\` / \`post\_cp\` congestion\-pricing indicators\. Bus profile observations are now calendar\-expanded to matching study\-period dates so the January 5, 2025 policy boundary is handled directly in this notebook rather than deferred downstream\. Taxi, Green Taxi, and FHVHV temporal standardization remain in Section 1\.2\.3 because those datasets require significant aggregation and restructuring to remain computationally manageable at scale\. With temporal standardization now strengthened across the non\-TLC mobility sources, the next step is spatial standardization, where Traffic, Bus, and Subway observations will be aligned to the project's canonical Taxi Zone geography while Bridge & Tunnel remains a system\-level mobility signal\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>